In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import h5py
from datetime import datetime
from metpy.calc import wind_components,wind_direction,wind_speed
from metpy.units import units

In [2]:
def format_datetime64(val, format_str='%Y-%m-%d %H:%M:%S'):
    dt = pd.to_datetime(str(val))
    return dt.strftime(format_str)

def get_daterange_str(ds):
    t = pd.to_datetime(ds.time.values,utc=True)
    return f'{t[0].strftime("%Y%m%d")}-{t[-1].strftime("%Y%m%d")}'

## Script settings

In [3]:
scope = 'H2'

project_root = '/work/bb1461/'
exp = 'v370_2030'

data_dir = '../data_in/'
data_out_dir = '../data_out/'

In [4]:
if scope == 'H2':
    scope_name = 'HEFEX II'
    obs_file = data_dir + f'hefex2_aws.h5'
    dt_from,dt_to = pd.to_datetime('2023-08-15', utc=True),pd.to_datetime('2023-09-10', utc=True)
elif scope == 'H3':
    scope_name = 'HEFEX III'
    obs_file = data_dir + f'hefex3_aws.h5'
    dt_from,dt_to = pd.to_datetime('2025-08-05', utc=True),pd.to_datetime('2025-09-05', utc=True)
else:
    print('Invalid scope!')

loc_file = data_dir + f'{scope}_Coords_w_cells.csv'

## Load and prepare location dimension

In [5]:
df_locs = pd.read_csv(loc_file).sort_values('Alt')
df_locs = df_locs[df_locs["Type"].str.startswith("AWS")]
df_locs

,New Name,Type,Lat,Lon,Comment,Alt,cell_id,topo
0,A255,AWS EC,46.810911,10.784094,MOMAA4. on debris,2556.0,34960.0,2654.089111
1,A260,AWS EC,46.809083,10.779727,MOMAA3,2605.0,37345.0,2690.791504
10,D262,AWS Met,46.808336,10.778370,TLOGGER,2623.0,37360.0,2722.822510
2,A267R,AWS EC,46.805415,10.776431,MOMAA9. Swiss line,2671.0,37442.0,2757.178711
4,A267,AWS EC,46.805913,10.775135,MOMAA8. Swiss line,2671.0,37433.0,2762.127930
3,A267L,AWS EC,46.806711,10.773102,MOMAA10. Swiss line,2672.0,37425.0,2762.057373
11,D269,AWS Wind,46.804296,10.773574,AWS,2692.0,37287.0,2792.991455
12,D273,AWS Met,46.801794,10.771454,TLOGGER,2725.0,35704.0,2798.971191
6,A275L,AWS EC,46.800647,10.768919,Grenoble,2741.0,35706.0,2802.588623
7,T275,AWS EC,46.800390,10.770579,Tower 2. Snowfox2,2744.0,35714.0,2810.582031


In [6]:
dim_loc = df_locs['New Name']
dim_loc = dim_loc.replace('Hintereis','STHE')

dim_loc_lat  = np.array(df_locs['Lat'])
dim_loc_lon  = df_locs['Lon']
dim_loc_type = df_locs['Type']
dim_loc_ele  = df_locs['Alt']

## Extract time series of each station for each variable

(variables have different names at each location)

In [7]:
# define column mapping for wind information
if scope == 'H2':
    wind_col_suffix = {
        'Hintereis': '_30',
        'A255'     : '_30_momaa',
        'A260'     : '_30_momaa',
        'A267R'    : '_30_momaa',
        'A267L'    : '_30_momaa',
        'A267'     : '_30_momaa',
        'D269'     : '_25',
        'T275'     : '_20_ec',
        'D281'     : '_25',
        'D293'     : '_25',
        'T303'     : '_20_ec',
        'A309'     : '_20_ec',
        'D316'     : '_25',
    }
elif scope == 'H3':
    wind_col_suffix = {
        'STHE' : '_30',
        'T303' : '_55',
    }
else:
    wind_col_suffix = {}


In [8]:
keys = h5py.File(obs_file, 'r').keys()
for k in keys:
    print(k,pd.DataFrame(pd.read_hdf(obs_file, key=k)).columns)

A255 Index(['T_10_ec', 'wspd_10_ec', 'wdir_10_ec', 'T_20_ec', 'wspd_20_ec',
       'wdir_20_ec', 'wspd_30_momaa_scalar', 'wspd_30_momaa_vector',
       'wdir_30_momaa', 'wdir_30_momaa_std', 'RH_20_momaa', 'T_20_momaa', 'p',
       'T_20_log'],
      dtype='object')
A260 Index(['T_10_ec', 'wspd_10_ec', 'wdir_10_ec', 'T_20_ec', 'wspd_20_ec',
       'wdir_20_ec', 'wspd_30_momaa_scalar', 'wspd_30_momaa_vector',
       'wdir_30_momaa', 'wdir_30_momaa_std', 'RH_20_momaa', 'T_20_momaa', 'p',
       'T_20_log'],
      dtype='object')
A267 Index(['T_10_ec', 'wspd_10_ec', 'wdir_10_ec', 'T_20_ec', 'wspd_20_ec',
       'wdir_20_ec', 'wspd_30_momaa_scalar', 'wspd_30_momaa_vector',
       'wdir_30_momaa', 'wdir_30_momaa_std', 'RH_20_momaa', 'T_20_momaa', 'p',
       'T_20_log'],
      dtype='object')
A267L Index(['T_10_ec', 'wspd_10_ec', 'wdir_10_ec', 'T_20_ec', 'wspd_20_ec',
       'wdir_20_ec', 'wspd_30_momaa_scalar', 'wspd_30_momaa_vector',
       'wdir_30_momaa', 'wdir_30_momaa_std', 'RH_20_moma

In [9]:
locs = np.array(df_locs['New Name'])

df_temp = pd.DataFrame()
df_pres = pd.DataFrame()
df_rh = pd.DataFrame()
df_wspd = pd.DataFrame()
df_wdir = pd.DataFrame()
df_u = pd.DataFrame()
df_v = pd.DataFrame()
df_wspd_avg = pd.DataFrame()
df_wdir_avg = pd.DataFrame()
df_u_avg = pd.DataFrame()
df_v_avg = pd.DataFrame()

keys = h5py.File(obs_file, 'r').keys()
for loc in locs:
    if loc not in keys:
        print(f'Location {loc} not in .h5 file')
        continue
    
    print('+', loc, end='\t')
    df = pd.DataFrame(pd.read_hdf(obs_file, key=loc)) # make sure it becomes a DataFrame, not Series
        
    # construct search terms for wind columns
    c_wspd, c_wdir = None, None
    if loc in wind_col_suffix.keys():       
        c_wspd = 'wspd' + wind_col_suffix[loc]
        c_wdir = 'wdir' + wind_col_suffix[loc]
        if c_wspd.endswith('momaa'):
            c_wspd = c_wspd + '_vector'        

    # determine column names for Temp, RH, Pres, Wind
    col_t, col_p, col_rh, col_wspd, col_wdir = None,None,None,None,None
    for c in df.columns:
        if c.startswith('T_20_log'):
            col_t = c
        elif (c.startswith('T_2') or c.startswith('T_15')) and col_t is None:
            # fallback for temp
            col_t = c
        elif c.endswith('_press'):
            df[c] = df[c] * 10  # cell pressure from irgason (HF)            
            col_p = c
        elif c.startswith('P') or c.startswith('p'):
            col_p = c
        elif c.startswith('RH_2') or c.startswith('RH_15'):
            col_rh = c
        elif c_wspd is not None and (c == c_wspd or c == 'FF'):
            col_wspd = c
        elif c_wdir is not None and c == c_wdir:
            col_wdir = c

    print('TEMP:',col_t, end='\t')
    print('PRES:',col_p, end='\t')
    print('RH:'  ,col_rh, end='\t')
    print('WSPD:',col_wspd, end='\t')
    print('WDIR:',col_wdir)

    df_hr = df[df.index.minute == 0]
    if col_t is not None:
        df_temp = df_temp.join(df_hr[col_t].dropna().rename(loc), how="outer")
    if col_p is not None:
        df_pres = df_pres.join(df_hr[col_p].dropna().rename(loc), how="outer")
    if col_rh is not None:
        df_rh = df_rh.join(df_hr[col_rh].dropna().rename(loc), how="outer")
    if col_wspd is not None:
        df_wspd = df_wspd.join(df_hr[col_wspd].dropna().rename(loc), how="outer")
    if col_wdir is not None:
        df_wdir = df_wdir.join(df_hr[col_wdir].dropna().rename(loc), how="outer")

    # calc wind components
    if col_wspd is not None and col_wdir is not None:
        df = df[[col_wspd,col_wdir]]
        df['u'], df['v'] = wind_components(df[col_wspd].values * units('m/s'),
                                           df[col_wdir].values * units.deg)
        df_hr = df[df.index.minute == 0]
        df_u = df_u.join(df_hr['u'].dropna().rename(loc), how="outer")
        df_v = df_v.join(df_hr['v'].dropna().rename(loc), how="outer")
        
        df_30min = df.resample('30min', origin='end_day').mean()
        df_30min['wspd'] = wind_speed(df_30min['u'].values * units('m/s'), df_30min['v'].values * units('m/s'))
        df_30min['wdir'] = wind_direction(df_30min['u'].values * units('m/s'), df_30min['v'].values * units('m/s'))
                
        df_u_avg = df_u_avg.join(df_30min['u'].dropna().rename(loc), how="outer")
        df_v_avg = df_v_avg.join(df_30min['v'].dropna().rename(loc), how="outer")
        df_wspd_avg = df_wspd_avg.join(df_30min['wspd'].dropna().rename(loc), how="outer")
        df_wdir_avg = df_wdir_avg.join(df_30min['wdir'].dropna().rename(loc), how="outer")

+ A255	TEMP: T_20_log	PRES: p	RH: RH_20_momaa	WSPD: wspd_30_momaa_vector	WDIR: wdir_30_momaa
+ A260	TEMP: T_20_log	PRES: p	RH: RH_20_momaa	WSPD: wspd_30_momaa_vector	WDIR: wdir_30_momaa
+ D262	TEMP: T_20_log	PRES: None	RH: None	WSPD: None	WDIR: None
+ A267R	TEMP: T_20_log	PRES: p	RH: RH_20_momaa	WSPD: wspd_30_momaa_vector	WDIR: wdir_30_momaa
+ A267	TEMP: T_20_log	PRES: p	RH: RH_20_momaa	WSPD: wspd_30_momaa_vector	WDIR: wdir_30_momaa
+ A267L	TEMP: T_20_log	PRES: p	RH: RH_20_momaa	WSPD: wspd_30_momaa_vector	WDIR: wdir_30_momaa
+ D269	TEMP: T_20_log	PRES: None	RH: RH_20_aws	WSPD: FF	WDIR: wdir_25
+ D273	TEMP: T_20_log	PRES: None	RH: None	WSPD: None	WDIR: None
Location A275L not in .h5 file
+ T275	TEMP: T_20_log	PRES: p	RH: RH_20_sf	WSPD: wspd_20_ec	WDIR: wdir_20_ec
Location A275R not in .h5 file
+ D281	TEMP: T_20_log	PRES: None	RH: RH_20_aws	WSPD: FF	WDIR: wdir_25
+ D285	TEMP: T_20_log	PRES: None	RH: None	WSPD: None	WDIR: None
+ D293	TEMP: T_20_log	PRES: None	RH: RH_20_aws	WSPD: FF	WDIR: 

In [10]:
def adjust_df(df):
    return df.reindex(locs,axis=1)[dt_from:dt_to]

df_temp = adjust_df(df_temp)
df_pres = adjust_df(df_pres)
df_rh   = adjust_df(df_rh)
df_u    = adjust_df(df_u)
df_v    = adjust_df(df_v)
df_wspd = adjust_df(df_wspd)
df_wdir = adjust_df(df_wdir)

df_u_avg = adjust_df(df_u_avg)
df_v_avg = adjust_df(df_v_avg)
df_wspd_avg = adjust_df(df_wspd_avg)
df_wdir_avg = adjust_df(df_wdir_avg)

In [11]:
# Filter data_in that seems to be off...
if scope == 'H2':
    df_temp.loc[pd.to_datetime('2023-09-01 00:00', utc=True):,'D281'] = np.nan
    df_rh.loc[pd.to_datetime('2023-08-17 00:00', utc=True):,'Hintereis'] = np.nan

## Convert DataFrames to DataArrays

In [12]:
def create_data_array(df, name, long_name, standard_name, unit, time_dim='time'):
    # get time from dataset
    time = df.index
    time_naive = [dt.replace(tzinfo=None) for dt in time]
    if time_dim.endswith('_avg'):
        t_res = '30min averages (origin is the last value of the timeseries)'
    else:
        t_res = 'Hourly instantaneous values'

    # get data_in for variable (all cell IDs)
    data = df.to_numpy()

    # create DataArray
    xr_data = xr.DataArray(
        data,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            "location": ("location", dim_loc, {"long_name": "Location name from field campaign"}),
            "lat": ("location", dim_loc_lat, {"long_name": "Latitude", "units": "degrees_north"}),
            "lon": ("location", dim_loc_lon, {"long_name": "Longitude", "units": "degrees_east"}),
            "type": ("location", dim_loc_type, {"long_name": "Type of measurement"}),
            "ele": ("location", dim_loc_ele, {"long_name": "Elevation of location", "units": "meters"})
        },
        dims=[time_dim, "location"],
        name=name,
        attrs={
            "long_name": long_name,
            "standard_name": standard_name,
            "units": unit,
            "temporal_resolution": t_res,
        }
    )
    return xr_data

## Build dataset

In [13]:
# Combine DataArrays
ds_out = xr.Dataset({
    't_2m'  : create_data_array(df_temp, 't_2m', 'temperature in 2m', 't_2m', 'degC'),
    'rh_2m' : create_data_array(df_rh,   'rh_2m', 'relative humidity in 2m', 'rh_2m','percent'),
    'pres'  : create_data_array(df_pres, 'pres',  'surface pressure', 'surface_air_pressure', 'hPa'),  
    'u'     : create_data_array(df_u, 'u',  'Zonal wind in 2m or 3m (station dependent)', 'eastward_wind', 'm s-1'),
    'v'     : create_data_array(df_v, 'v',  'Meridional wind in 2m or 3m (station dependent)', 'northward_wind', 'm s-1'),
    'wspd'  : create_data_array(df_wspd, 'wspd',  'wind speed in 2m or 3m (station dependent)', '', 'm s-1'),  
    'wdir'  : create_data_array(df_wdir, 'wdir',  'wind direction in 2m or 3m (station dependent)', '', 'deg'),  
    
    'u_avg'     : create_data_array(df_u_avg, 'u_avg',  'Zonal wind in 2m or 3m (station dependent)', 'eastward_wind', 'm s-1', 'time_avg'),
    'v_avg'     : create_data_array(df_v_avg, 'v_avg',  'Meridional wind in 2m or 3m (station dependent)', 'northward_wind', 'm s-1', 'time_avg'),
    'wspd_avg'  : create_data_array(df_wspd_avg, 'wspd_avg',  'wind speed in 2m or 3m (station dependent)', '', 'm s-1', 'time_avg'),  
    'wdir_avg'  : create_data_array(df_wdir_avg, 'wdir_avg',  'wind direction in 2m or 3m (station dependent)', '', 'deg', 'time_avg'), 
})

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="1h")
ds_out = ds_out.reindex(time=time_full)

ds_out = ds_out.sortby("time_avg")
ds_out = ds_out.sel(time_avg=~ds_out.indexes["time_avg"].duplicated())
time_full = pd.date_range(ds_out.time_avg.min().values, ds_out.time_avg.max().values, freq="30min")
ds_out = ds_out.reindex(time_avg=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Observational data from {scope_name} AWS",
    "institution": "Humboldt-Universität zu Berlin",
    "source": f"Aggregation of various datasets from {scope_name}",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": scope_name,
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Full observational dataset on request"
}

# Write
output_file = data_out_dir + f'HEFEX{scope[1]}_Obs_AWS_L1_1h_avg30min_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/HEFEX2_Obs_AWS_L1_1h_avg30min_20230816-20230910.nc.


In [14]:
ds_out

<xarray.Dataset> Size: 1MB
Dimensions:   (time: 593, location: 19, time_avg: 1110)
Coordinates:
  * time      (time) datetime64[ns] 5kB 2023-08-16T08:00:00 ... 2023-09-10
  * location  (location) object 152B 'A255' 'A260' 'D262' ... 'A309' 'D316'
  * time_avg  (time_avg) datetime64[ns] 9kB 2023-08-16T08:00:00 ... 2023-09-0...
    lat       (location) float64 152B 46.81 46.81 46.81 ... 46.79 46.79 46.8
    lon       (location) float64 152B 10.78 10.78 10.78 ... 10.75 10.75 10.74
    type      (location) object 152B 'AWS EC' 'AWS EC' ... 'AWS EC' 'AWS Wind'
    ele       (location) float64 152B 2.556e+03 2.605e+03 ... 3.09e+03 3.16e+03
Data variables:
    t_2m      (time, location) float64 90kB nan nan nan nan ... nan nan nan nan
    rh_2m     (time, location) float64 90kB 68.6 nan nan nan ... nan nan nan nan
    pres      (time, location) float64 90kB 756.8 nan nan nan ... nan nan nan
    u         (time, location) float64 90kB 1.206 nan nan nan ... nan nan nan
    v         (time, location) float64 90kB 3.03 nan nan nan ... nan nan nan nan
    wspd      (time, location) float64 90kB 3.261 nan nan nan ... nan nan nan
    wdir      (time, location) float64 90kB 201.7 nan nan nan ... nan nan nan
    u_avg     (time_avg, location) float64 169kB 1.819 nan nan ... nan nan nan
    v_avg     (time_avg, location) float64 169kB 2.696 nan nan ... nan nan nan
    wspd_avg  (time_avg, location) float64 169kB 3.252 nan nan ... nan nan nan
    wdir_avg  (time_avg, location) float64 169kB 214.0 nan nan ... nan nan nan
Attributes:
    title:        Observational data from HEFEX II AWS
    institution:  Humboldt-Universität zu Berlin
    source:       Aggregation of various datasets from HEFEX II
    history:      Created 2026-04-15
    contact:      alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-...
    campaign:     HEFEX II
    StartTime:    2023-08-16 08:00:00
    StopTime:     2023-09-10 00:00:00
    comment:      Full observational dataset on request